In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
RAIZ = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, RAIZ)
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from src.comun.rutas import DATA_GENERADO, MODELOS
M = json.load(open(os.path.join(MODELOS, "metricas.json"), encoding="utf-8")) if os.path.isfile(os.path.join(MODELOS, "metricas.json")) else {}
for f in os.listdir(MODELOS):
    if f.startswith("metricas_") and f.endswith(".json") and not f.startswith(("metricas_rag", "metricas_agente")):
        M[f[len("metricas_"):-5]] = json.load(open(os.path.join(MODELOS, f), encoding="utf-8"))
def tabla(d, dec=3):
    return pd.DataFrame(d).T.round(dec) if isinstance(d, dict) else pd.DataFrame(d)


# 02. Segmentación con K-Prototypes y nombres por nivel

K-Prototypes (variables mixtas) frente a K-Means y a reglas por umbral. El nombre de cada segmento sale del ranking de su salud financiera mediana.

In [2]:
S = M['segmentacion']; print('k elegido:', S['k']); tabla(S['tabla_k'])

k elegido: 3


,costo,silueta_gower,estabilidad_ari,tamano_min,puntaje
3,4665.778,0.205,0.744,68.0,0.872
4,4261.577,0.175,0.651,53.0,0.702
5,3996.510,0.126,0.673,49.0,0.514
6,3776.340,0.158,0.647,20.0,0.632
7,3612.372,0.083,0.592,20.0,0.296
8,3450.172,0.129,0.516,15.0,0.445


In [3]:
print('Silueta de Gower:', S['silueta_gower']); tabla(S['ari_vs_etiquetas'])

Silueta de Gower: {'kprototypes': 0.20482109545241844, 'kmeans_onehot': 0.20363475441334158, 'reglas_2x2': 0.0579717505199914}


,arquetipo,segmento_real,perfil_2x2
kprototypes,0.253,0.292,0.017
kmeans_onehot,0.218,0.277,0.016
reglas_2x2,0.022,0.037,0.184


## Los segmentos y su nombre

In [4]:
d = tabla(S['descripcion'], 2); d[['nivel', 'n', 'salud_mediana', 'ingreso_medio', 'cv_ingreso', 'ahorro_pct', 'gasto_sobre_ingreso', 'dias_rojo_3m', 'colchon_observado']]

,nivel,n,salud_mediana,ingreso_medio,cv_ingreso,ahorro_pct,gasto_sobre_ingreso,dias_rojo_3m,colchon_observado
0,luchador,68,45.9,1376.817,0.6,0.057,1.123,17.5,0.729
1,constructor,165,70.3,1257.466,0.436,0.251,0.8,0.0,1.645
2,campeon,267,82.9,2571.842,0.017,0.31,0.732,0.0,1.701


## Composición de los niveles por grupo (control de sesgo)
Si un nivel estuviera dominado por una región o una edad, sería una señal de sesgo.

In [5]:
for g, t in S['composicion_de_niveles_por_grupo'].items():
    print(g); display(tabla(t, 2))

region


,campeon,constructor,luchador
centro,0.54,0.33,0.13
lima,0.53,0.33,0.14
norte,0.56,0.31,0.13
oriente,0.48,0.35,0.18
sur,0.54,0.34,0.12


rango_edad


,campeon,constructor,luchador
18-24,0.54,0.35,0.11
25-34,0.52,0.35,0.12
35-44,0.51,0.33,0.16
45-54,0.57,0.26,0.17
55+,0.54,0.37,0.09


arquetipo


,campeon,constructor,luchador
base_comision,0.88,0.07,0.05
formal_quincena,0.94,0.03,0.03
independiente,0.14,0.63,0.24
mixto,0.44,0.40,0.16


**Arranque en frío** (predecir el nivel solo con lo declarado): precisión de validación cruzada frente a la clase mayoritaria.

In [6]:
print(round(S['arranque_en_frio_precision'], 3), 'frente a', round(S['referencia_clase_mayoritaria'], 3))

0.738 frente a 0.534
